In [4]:
import fitz  # PyMuPDF
import cv2
import numpy as np
import os
from PIL import Image
import imagehash

def extract_annotations(doc, color='red'):
    annotations_info = []
    color_map = {'red': ([0, 0, 250], [10, 10, 255])}  # Define BGR ranges for red

    for page_number in range(len(doc)):
        page = doc[page_number]
        pix = page.get_pixmap()  # Get the pixmap for the current page

        if pix.samples:  # Ensure the pixmap data is not empty
            # Convert pixmap to an OpenCV image
            image = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)  # Convert from RGB to BGR

            # Retrieve the color ranges for the specified color
            lower_color, upper_color = color_map[color] if color in color_map else ([0, 0, 0], [255, 255, 255])
            lower_red = np.array(lower_color)
            upper_red = np.array(upper_color)

            # Create a mask to isolate the areas of interest
            mask = cv2.inRange(image, lower_red, upper_red)
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # Process each contour found
            for rect in [cv2.boundingRect(cnt) for cnt in contours]:
                x0, y0, w, h = rect
                x1, y1 = x0 + w, y0 + h
                annotations_info.append((page_number + 1, x0, y0, x1, y1))

    return annotations_info



def remove_border(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        # Find the largest contour which should be the border
        contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(contour)
        img = img[y:y+h, x:x+w]
    return img, (x, y, w, h)

def extract_images(doc, annotations_info, output_folder, remove_border_flag=False):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    output_images = []
    new_annotations_info = []
    for idx, (page_number, x0, y0, x1, y1) in enumerate(annotations_info):
        page = doc.load_page(page_number - 1)
        clip = fitz.Rect(x0, y0, x1, y1)
        pix = page.get_pixmap(clip=clip)
        img = cv2.imdecode(np.frombuffer(pix.tobytes(), dtype=np.uint8), cv2.IMREAD_COLOR)
        if remove_border_flag:
            img, (border_x, border_y, border_w, border_h) = remove_border(img)
            new_x0 = x0 + border_x
            new_y0 = y0 + border_y
            new_x1 = x0 + border_x + border_w
            new_y1 = y0 + border_y + border_h
            new_annotations_info.append((page_number, new_x0, new_y0, new_x1, new_y1))
        else:
            new_annotations_info.append((page_number, x0, y0, x1, y1))
        output_images.append(img)
        img_path = os.path.join(output_folder, f"extracted_image_{idx + 1}.png")
        cv2.imwrite(img_path, img)
    return output_images, new_annotations_info

def adjust_annotations_for_pdf2(original_annotations, new_annotations):
    adjusted_annotations = []
    for (page_number, x0, y0, x1, y1), (new_page_number, new_x0, new_y0, new_x1, new_y1) in zip(original_annotations, new_annotations):
        adjusted_annotations.append((page_number, new_x0, new_y0, new_x1, new_y1))
    return adjusted_annotations

input_path1 = "/home/emanmunir/detection/test2/small-pdf-boxes.pdf"
input_path2 = "/home/emanmunir/detection/test2/small-pdf.pdf"
doc1 = fitz.open(input_path1)
doc2 = fitz.open(input_path2)

annotations_info1 = extract_annotations(doc1)

output_folder1 = "extracted_images_pdf1"
output_folder2 = "extracted_images_pdf2"
output_images1, new_annotations_info1 = extract_images(doc1, annotations_info1, output_folder1, remove_border_flag=True)
adjusted_annotations_info2 = adjust_annotations_for_pdf2(annotations_info1, new_annotations_info1)
output_images2, _ = extract_images(doc2, adjusted_annotations_info2, output_folder2)

def compute_hash_similarity(img1, img2):
    img1_pil = Image.fromarray(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
    img2_pil = Image.fromarray(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
    hash1 = imagehash.phash(img1_pil)
    hash2 = imagehash.phash(img2_pil)
    similarity = 1 - (hash1 - hash2) / len(hash1.hash) ** 2
    return similarity

similarities = []
for img1, img2 in zip(output_images1, output_images2):
    score = compute_hash_similarity(img1, img2)
    similarities.append(score)

annotations_info1, similarities


([(1, 83, 336, 219, 406),
  (2, 324, 294, 442, 377),
  (3, 88, 478, 227, 627),
  (3, 348, 70, 494, 210),
  (4, 330, 450, 469, 592),
  (4, 85, 76, 263, 211),
  (5, 371, 135, 513, 257),
  (6, 364, 152, 489, 311),
  (7, 90, 636, 251, 724),
  (7, 289, 634, 520, 737),
  (7, 247, 481, 384, 552),
  (7, 405, 477, 555, 570),
  (7, 448, 109, 541, 182),
  (8, 370, 252, 548, 355),
  (8, 154, 252, 327, 351)],
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0])